# 形状变换 (Reshaping)

深度学习中最常用的操作之一，用于调整数据维度。

- tensor.view(shape): 改变形状（要求内存连续）。
- tensor.reshape(shape): 改变形状（如果内存不连续会自动复制，比 view 更通用）。
- tensor.flatten(): 将张量展平为一维。
- tensor.unsqueeze(dim): 增加一个维度（例如将 [N] 变为 [N, 1]）。
- tensor.squeeze(dim): 去除维度为 1 的维度。
- tensor.permute(dims): 交换维度（例如 [Batch, Channel, H, W] 转为 [Batch, H, W, Channel]）。
- tensor.T: 转置（二维矩阵）。

## 🧬 改变张量形状的核心操作

| 操作 | 核心功能 | 内存连续性要求 | 内存共享情况 |
| :--- | :--- | :--- | :--- |
| `view` | 改变形状 | 必须连续 | 总是共享 |
| `reshape` | 改变形状 | 不要求（自动处理） | 连续则共享，不连续则不共享 |
| `permute` | 交换维度 | 操作后通常不连续 | 总是共享 |
| `transpose`/`.T` | 交换维度 | 操作后通常不连续 | 总是共享 |

💡 实践建议
- 日常开发首选 reshape：在大多数情况下，使用 reshape 是更安全、更省心的选择，它能帮你避免很多由张量不连续导致的潜在错误。
- 性能敏感时用 view：如果你确定张量是连续的，并且追求极致的性能（避免数据拷贝），可以使用 view。
- 组合操作：如果你需要使用 view，但又不确定张量是否连续，一个常见的做法是先调用 .contiguous()，例如 x.permute(0, 2, 3, 1).contiguous().view(...)

### tensor.view(shape)
功能：改变张量的形状。
特点：
- <font color='red'>要求原张量在内存中是连续（contiguous）的。</font>
- 如果张量不连续，调用 view 会报错。
- 它返回的是原张量的一个“视图”，<font color='red'>与原张量共享内存，修改其中一个会影响另一个。</font>


In [2]:
import torch

x = torch.randn(2,3,4) # 创建一个连续的张量
print(x)
y = x.view(6,4) # 成功改变形状
print(y)

tensor([[[-1.3735,  0.8884,  0.6763,  0.1588],
         [ 0.8472, -2.2705, -0.6488, -1.4990],
         [-0.1402, -1.6603, -0.1538,  1.7832]],

        [[-1.2876,  1.1907,  0.6756,  0.1800],
         [-0.5450,  1.1124,  1.8052,  0.3025],
         [ 0.1017,  0.2557, -0.8465, -0.6421]]])
tensor([[-1.3735,  0.8884,  0.6763,  0.1588],
        [ 0.8472, -2.2705, -0.6488, -1.4990],
        [-0.1402, -1.6603, -0.1538,  1.7832],
        [-1.2876,  1.1907,  0.6756,  0.1800],
        [-0.5450,  1.1124,  1.8052,  0.3025],
        [ 0.1017,  0.2557, -0.8465, -0.6421]])


### tensor.reshape(shape)
- 功能：改变张量的形状。
- 特点：<font color='red'>比 view 更通用、更安全。它不要求原张量必须是连续的。</font>
    - <font color='red'>如果张量不连续，reshape 会在内部自动调用 .contiguous() 进行数据复制，然后再改变形状。因此，它可能返回一个视图（共享内存），也可能返回一个副本（不共享内存）。</font>

In [4]:
x = torch.randn(2,3,4)
x_t = x.transpose(0,1) # 装置操作会使张量变得不连续
print(x_t.is_contiguous())
y = x_t.reshape(6,4) # 成功！reshape会自动处理不连续的情况
print(y.is_contiguous())

False
True


### tensor.flatten(start_dim=0, end_dim=-1)
- 功能：将张量的一部分或全部展平成一维。
- 特点：<font color='red'>这是 view 或 reshape 的一个特例，专门用于展平操作，语义更清晰。</font>
    - 默认情况下，它会将所有维度展平。你也可以通过 start_dim 和 end_dim 参数指定要展平的范围。

In [6]:
x = torch.randn(2,3,4)
print(x)
y = x.flatten() # 展平所有维度，得到形状(24,)
print(y)
z = x.flatten(1,2) # 展平后两个维度，得到(2,12)

tensor([[[-9.7554e-02,  5.3309e-01,  2.6274e-01, -8.3427e-02],
         [ 1.3210e+00, -6.2292e-02,  1.3634e+00, -2.5055e-01],
         [ 1.1276e+00,  1.6272e+00, -9.3522e-01, -1.0617e+00]],

        [[ 1.3025e-01, -2.3811e+00,  1.4115e-01, -1.1694e+00],
         [ 1.8021e+00,  8.1708e-01, -1.1633e+00,  2.1419e-03],
         [ 8.4021e-02, -1.8865e+00,  9.1805e-01, -2.1736e+00]]])
tensor([-9.7554e-02,  5.3309e-01,  2.6274e-01, -8.3427e-02,  1.3210e+00,
        -6.2292e-02,  1.3634e+00, -2.5055e-01,  1.1276e+00,  1.6272e+00,
        -9.3522e-01, -1.0617e+00,  1.3025e-01, -2.3811e+00,  1.4115e-01,
        -1.1694e+00,  1.8021e+00,  8.1708e-01, -1.1633e+00,  2.1419e-03,
         8.4021e-02, -1.8865e+00,  9.1805e-01, -2.1736e+00])


### tensor.unsqueeze(dim)
- 功能：在指定位置 dim 插入一个大小为 1 的新维度。
- 特点：<font color='red'>常用于为张量增加批次（batch）维度或通道（channel）维度，以满足某些操作的输入要求。</font>

In [ ]:
x = torch.randn(3) # 形状为(3,)

y = x.unsqueeze(0) # 在第0维 增加维度，形状为(1,3)
z = x.unsqueeze(1) # 在第1维 增加维度，形状为(3,1)



### tensor.squeeze(dim=None)
- 功能：移除所有大小为 1 的维度。
- 特点：与 unsqueeze 相反。如果指定了 dim 参数，则只移除该指定位置上大小为 1 的维度。

In [7]:
x = torch.randn(1,3,1,2)

y = x.squeeze() # 移除所有大小为1的维度，形状变为(3,2)
print(y.shape)

z = x.squeeze(0) # 只移除第0维的大小为1的维度，形状变为(3,1,2)
print(z.shape)

torch.Size([3, 2])
torch.Size([3, 1, 2])


### tensor.permute(*dims)
- 功能：根据指定的维度顺序 dims 重新排列张量的所有维度。
- 特点：功能强大，可以任意交换多个维度。<font color='red'>需要注意的是，permute 操作通常会使张量变得不连续。</font>

In [ ]:
x = torch.randn(2,3,4,5) # 形状为(batch,c,h,w)
y = x.permute(0,2,3,1) # 交换维度，形状变为(batch,h,w,c)

### tensor.T
- 功能：对张量进行转置。
- 特点：<font color='red'>这是 transpose 的一个便捷属性，仅适用于二维张量，等价于 tensor.transpose(0, 1)。和 permute 一样，它也会使张量变得不连续。</font>

In [ ]:
x = torch.randn(3,4) # 形状为(3,4)
y = x.T  # 装置后形状为(4,3)

### tensor.transpose(dim0, dim1) 是 PyTorch 中用于交换张量两个指定维度的操作。

- 它是处理高维数据时非常基础和重要的工具，例如在深度学习模型中经常需要调整数据的维度顺序（如将图像数据从 [Batch, Channel, Height, Width] 调整为 [Batch, Height, Width, Channel]）。

📝 核心用法与特点
- 功能：交换张量的 dim0 和 dim1 这两个维度。
- 特点：
    - <font color='red'>返回视图（View）：transpose 操作不会复制底层数据，而是返回原张量的一个“视图”。这意味着新张量和原张量共享同一块内存，修改其中一个会影响另一个。</font>
    - <font color='red'>导致不连续（Non-contiguous）</font>：交换维度后，张量在内存中的存储通常会变得不连续。这对于某些后续操作（如 view）可能会有影响。
- 与 .T 的关系：.T 是 transpose 的一个便捷属性，但它仅适用于二维张量，等价于 tensor.transpose(0, 1)。对于三维及以上的张量，必须使用 transpose(dim0, dim1) 方法来明确指定要交换的维度。

In [8]:
# 创建一个三维张量，形状为 (2, 3, 4)
x = torch.randn(2, 3, 4)
print(f"原始张量形状: {x.shape}")  # torch.Size([2, 3, 4])

# 交换第1维（dim=1）和第2维（dim=2）
x_transposed = x.transpose(1, 2)
print(f"转置后张量形状: {x_transposed.shape}")  # torch.Size([2, 4, 3])

# 验证共享内存
print(f"共享内存:{x.storage().data_ptr() == x_transposed.storage().data_ptr}")

原始张量形状: torch.Size([2, 3, 4])
转置后张量形状: torch.Size([2, 4, 3])
共享内存:False


/var/folders/lq/hmdtq6_x06d32l7lfmd0g7gr0000gn/T/ipykernel_43191/2961411172.py:10: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  print(f"共享内存:{x.storage().data_ptr() == x_transposed.storage().data_ptr}")


#### ⚠️ 重要注意事项
- 由于 transpose 操作会使张量变得不连续，这可能会导致后续一些要求张量连续的操作（例如 view）失败。
- 解决方案：如果你在对张量进行 transpose 后，需要使用 view 等方法，通常需要先调用 .contiguous() 方法，使张量在内存中重新变为连续。

In [ ]:
# 假设 x_transposed 是经过 transpose 操作的张量
# 直接 view 可能会报错
# y = x_transposed.view(-1)  # 可能引发 RuntimeError

# 正确的做法是先 contiguous()
y = x_transposed.contiguous().view(-1) # 确保内存连续后再改变形状